# Clustering (et exploitation des données géographiques)

Dans cette partie nous exploitons les données géographiques et réalisons le clustering. 

Nous exportons des statistiques par commune pour croisement avec les données de vote. 

In [67]:
from BDTopo_fonctions import load_gpkg
gdf=load_gpkg("Sitadel/df_clustering_fulldep_1000m3.gpkg")

Téléchargement depuis mgarbe/Sitadel/df_clustering_fulldep_1000m3.gpkg ...
Erreur lors du chargement : An error occurred (404) when calling the HeadObject operation: Not Found


In [ ]:
#PB DANS GDF : ON A DES OBS EN DOUBLE (VIENT SUREMENT DE TEMP_STOCK2013 ??)

In [57]:
import importlib
import clustering_fonctions
importlib.reload(clustering_fonctions)
from clustering_fonctions import run_dbscan_parallele
temp=run_dbscan_parallele(gdf,300,15)

248674    MULTIPOLYGON Z (((4.36496 45.42478 634.7, 4.36...
249155    MULTIPOLYGON Z (((4.36496 45.42478 634.7, 4.36...
249219    MULTIPOLYGON Z (((4.36496 45.42478 -1000, 4.36...
249307    MULTIPOLYGON Z (((4.36496 45.42478 -1000, 4.36...
249325    MULTIPOLYGON Z (((4.36496 45.42478 -1000, 4.36...
249434    MULTIPOLYGON Z (((4.36496 45.42478 634.7, 4.36...
249529    MULTIPOLYGON Z (((4.36496 45.42478 634.7, 4.36...
252125    MULTIPOLYGON Z (((4.36387 45.42477 632, 4.3644...
252126    MULTIPOLYGON Z (((4.36743 45.42375 647.4, 4.36...
252128    MULTIPOLYGON Z (((4.36993 45.42183 665.6, 4.36...
252391    MULTIPOLYGON Z (((4.36784 45.42237 650.2, 4.36...
252392    MULTIPOLYGON Z (((4.36993 45.42183 665.6, 4.36...
252393    MULTIPOLYGON Z (((4.37109 45.42276 668.6, 4.37...
253769                             POINT (4.36601 45.42121)
253784    MULTIPOLYGON Z (((4.36496 45.42478 9999, 4.365...
Name: geometry, dtype: geometry

In [59]:
# Cluster_id présents dans Sitadel
sitadel_ids = temp.loc[temp["Base"]=="Sitadel", "cluster_id_2014"].unique()

# Comptage des occurrences de tous les bâtiments pour ces cluster_id
vc = temp["cluster_id_2025"].value_counts()

# Filtrer uniquement les clusters présents dans Sitadel
vc_sitadel = vc[vc.index.isin(sitadel_ids)]

# Nombre de clusters de taille < 5
vc_sitadel[vc_sitadel.index > 0]

cluster_id_2025
1400009    1816
3400026    1196
3400047     987
7800013     925
9200003     871
           ... 
4200073      15
5300041      15
6000066      15
3200016      15
6200164      13
Name: count, Length: 478, dtype: int64

In [60]:
# CLUSTER TOTAL, TOUS BAT EXISTANTS OU AUTORISES, ANNEE PAR ANNEE 
import pandas as pd
temp_sorted = temp.sort_values("Annee_REF")
results = []
for annee in sorted(temp_sorted["Annee_REF"].unique()):
    mask = temp_sorted["Annee_REF"] <= annee  # toutes les années <= annee
    cluster_col = temp_sorted.loc[mask, "cluster_id"]
    pct_cluster = (cluster_col > 0).sum() / len(cluster_col) * 100
    results.append({
        "Annee": annee,
        "pct_cluster_cumule": pct_cluster
    })
df_cumule = pd.DataFrame(results)
print(df_cumule)

    Annee  pct_cluster_cumule
0    2013           53.422197
1    2014           53.156880
2    2015           52.869758
3    2016           52.608944
4    2017           52.403900
5    2018           52.280514
6    2019           52.985531
7    2020           52.872675
8    2021           52.836146
9    2022           52.791348
10   2023           52.752931
11   2024           52.711526
12   2025           52.668794


In [49]:
#PC QUI EN 2025 SONT DANS UN CLUSTER
temp[temp["Base"]=="Sitadel"].groupby("Annee_REF")["cluster_id"].apply(lambda x: (x > 0).sum() / len(x) * 100)

Annee_REF
2014    63.523866
2015    64.585698
2016    65.986029
2017    64.257257
2018    66.337209
2019    66.933006
2020    63.586531
2021    67.021985
2022    68.193225
2023    66.034875
2024    61.597938
2025    64.472050
Name: cluster_id, dtype: float64

In [50]:
#PC QUI SONT DANS UN CLUSTER AU MOMENT DE AUTORISATION 

temp_sitadel = temp[(temp["Base"] == "Sitadel")]

for annee in sorted(temp_sitadel["Annee_REF"].unique()):
    col = f"cluster_id_{annee}"
    mask = temp_sitadel["Annee_REF"] == annee
    cluster_col = temp_sitadel.loc[mask, col]
    pct_cluster = (cluster_col > 0).sum() / len(cluster_col) * 100
    print(f"Année {annee}: {pct_cluster:.1f}% des bâtiments clusterisés")

Année 2014: 58.2% des bâtiments clusterisés
Année 2015: 58.7% des bâtiments clusterisés
Année 2016: 60.4% des bâtiments clusterisés
Année 2017: 59.6% des bâtiments clusterisés
Année 2018: 62.2% des bâtiments clusterisés
Année 2019: 64.0% des bâtiments clusterisés
Année 2020: 61.7% des bâtiments clusterisés
Année 2021: 64.9% des bâtiments clusterisés
Année 2022: 66.9% des bâtiments clusterisés
Année 2023: 65.7% des bâtiments clusterisés
Année 2024: 61.2% des bâtiments clusterisés
Année 2025: 64.5% des bâtiments clusterisés


In [43]:
annees = sorted(temp["Annee_REF"].unique())
result_list = []
for annee in annees:
    col = f"cluster_id_{annee}"
    if col in temp.columns:
        # compter les clusters uniques non-négatifs
        nb_clusters = temp[col][temp[col] > 0].nunique()
        result_list.append({"Annee_REF": annee, "nb_clusters": nb_clusters})

df_clusters_par_annee = pd.DataFrame(result_list)
print(df_clusters_par_annee)

    Annee_REF  nb_clusters
0        2013         4194
1        2014         4243
2        2015         4270
3        2016         4300
4        2017         4323
5        2018         4337
6        2019         4750
7        2020         4801
8        2021         4807
9        2022         4815
10       2023         4829
11       2024         4848
12       2025         4862
